# 32장 · LCEL과 LangGraph의 상태 전달

같은 검색·답변 흐름도 LCEL에서는 함수 사이의 반환값으로, LangGraph에서는 상태 갱신으로 연결할 수 있습니다.

같은 질문과 문서, 검색 함수, k를 고정해 LCEL과 LangGraph를 실행합니다. 최종 답변이 같더라도 중간 상태의 필드를 누가 다음 단계로 전달하는지는 두 구현에서 다릅니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain-core 1.6.12, langgraph 1.2.16입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## Runnable과 그래프 노드의 상태 전달 비교

검색을 거치면 documents가 생기고 답 작성 단계는 거기서 첫 문장을 꺼냅니다. LCEL은 앞 함수의 반환값 전체를 다음 함수에 넘깁니다. Graph는 노드의 부분 반환값을 상태에 반영합니다. 두 코드가 공유하는 자료는 로컬 키워드 벡터로 찾으며 모델은 호출하지 않습니다.


### 셀 01 · LCEL 함수의 상태 반환

검색 후에도 question이 남도록 입력 딕셔너리와 새 documents를 합칩니다.


In [1]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]

from langchain_core.runnables import RunnableLambda

def search_change(state, *, k=1):
    return {'documents': retrieve(state['question'], k)}

def answer_change(state):
    docs = state['documents']
    return {'answer': docs[0]['text'] if docs else '제공된 자료에서 확인할 수 없습니다.'}

chain = RunnableLambda(lambda state: {**state, **search_change(state)}) | RunnableLambda(lambda state: {**state, **answer_change(state)})
request = {'question': '대출 기간은?', 'documents': [], 'answer': ''}
left = chain.invoke(request)
print(left['answer'])

도서 대출 기간은 14일입니다.


검색 문장이 그대로 답에 들어갔습니다. 두 번째 함수에 전달되는 값에는 원래 질문과 새 문서 목록이 모두 있습니다.


### 같은 입력과 같은 검색 함수로 비교하기

<p>두 구현 모두 대출 질문, 같은 세 문서, k=1에서 출발합니다. search_change는 documents 한 칸을, answer_change는 answer 한 칸을 반환합니다. 이 두 변경분을 누가 기존 상태와 합치는지 따라가 보세요.</p>


### LCEL 래퍼에서 기존 키를 보존하는 위치

<p>파이프 오른쪽은 왼쪽이 반환한 값을 그대로 받습니다. 각 RunnableLambda에서 {**state, **변경분}을 돌려주므로 이전 question과 documents가 다음 단계에도 들어갑니다.</p>


### 셀 02 · StateGraph의 부분 갱신

LCEL은 기본값 k=1을 씁니다. 그래프의 검색 노드는 이 셀의 graph_k를 읽습니다. 먼저 두 값을 모두 1로 두고 실행하세요.


In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    documents: list[dict]
    answer: str

graph_k = 1

def graph_search(state):
    return search_change(state, k=graph_k)

builder = StateGraph(State)
builder.add_node('search', graph_search)
builder.add_node('compose', answer_change)
builder.add_edge(START, 'search')
builder.add_edge('search', 'compose')
builder.add_edge('compose', END)
graph = builder.compile()
right = graph.invoke(request)
print(right['answer'], right == left)
for name, state in [("LCEL", left), ("Graph", right)]:
    print(name, [doc["id"] for doc in state["documents"]])

도서 대출 기간은 14일입니다. True
LCEL ['library-1']
Graph ['library-1']


마지막 두 줄에 모두 library-1 하나가 나옵니다. graph_k를 2로 바꾸고 이 셀만 다시 실행하면 그래프 쪽에 library-2가 하나 더 생깁니다. 첫 문서를 읽어 만든 답은 여전히 14일이지만 상태 전체를 비교한 값은 False가 됩니다.

두 답이 같아도 검색 단계가 넘긴 값을 들여다보면 구현 차이가 보입니다. LCEL 함수에서는 기존 필드를 다음 사전에 직접 담았습니다. 그래프에서는 검색 노드가 documents만 돌려줘도 원래 question이 남습니다. 최종 문장만 테스트하면 이런 중간 값의 누락을 놓치기 쉽습니다.


### StateGraph가 노드 변경분을 반영하는 위치

<p>graph_search가 반환한 documents가 공유 상태에 합쳐진 뒤 compose가 실행됩니다. compose가 answer만 반환해도 최종 상태에는 검색한 문서가 남습니다. 셀 02는 답과 문서 ID를 두 구현에서 나란히 확인합니다.</p>


## stream()으로 노드 갱신 추적

마지막 문장만 비교하면 검색부터 달랐는지 알기 어렵습니다. 각 노드의 변경분을 살펴보고, 검색 ID가 같을 때만 답 작성과 인용 처리를 다음 후보로 봅니다.


### 과제 03 · stream()의 첫 노드 갱신

updates 모드에서 첫 갱신을 받아 검색된 첫 문서의 ID를 표시하세요. 셀 02에서 graph_k를 2로 바꾸었을 때도 첫 ID가 같은지 확인하세요. 두 번째 ID가 추가됐는지는 셀 02의 문서 목록에서 볼 수 있습니다.


In [3]:
# next(graph.stream(...))로 첫 갱신을 받으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>first = next(graph.stream(request, stream_mode=&#x27;updates&#x27;))
print(first[&#x27;search&#x27;][&#x27;documents&#x27;][0][&#x27;id&#x27;])</code></pre>
<p>실행 결과</p><pre><code>library-1</code></pre>
<p>library-1은 답을 쓴 뒤에 만들어진 ID가 아닙니다. search의 반환값에 이미 들어 있습니다. 같은 실험의 소요시간을 기록한다면 어느 함수 전후를 측정했는지도 함께 남기세요.</p>
</details>


### stream_mode="updates"의 첫 항목

<p>next()로 꺼낸 첫 갱신은 search라는 노드 이름 아래 documents를 담고 있습니다. 이 시점의 library-1은 검색의 반환값입니다. invoke가 끝난 뒤 받는 전체 상태와 구별해 읽으세요.</p>


### 스트림의 첫 갱신에서 어디까지 관찰했을까요?

애니메이션은 next()를 한 번 호출한 관찰자 시점입니다. 첫 항목은 실행 전체의 최종 상태가 아니라 search가 방금 반환한 변경분입니다.

1. stream · 모드 = updates
2. search · 갱신한 키 = documents
3. 첫 항목 · 바깥 키 = search
4. 첫 ID 출력 · documents[0]["id"] = library-1


## 돌아보기

셀 02의 graph_k를 2로 바꾸고 다시 실행하세요. 두 문서 목록에서 달라진 ID를 적고 답 문장이 같은 이유를 설명하세요. graph_k를 1로 되돌려 두 상태가 다시 같은지 확인하세요.
